# Preparing dataframes for pseudobulking analysis - Sample type Ovarian Cancer Samples only
## Using DC anndata object data_oc.h5ad 
* sum by sample, so for each DC subset (3 dataframes) made a dataframe that has genes as rows and integration_id (samples) as columns
* then make a metadata dataframe that has two columns --> integration_id and cancer type

## Using code from NR 2025-01-13_Pseudobulking.ipynb


In [4]:
#load packages
import scanpy as sc
import os
import pandas as pd

In [2]:
#change current directory 
os.chdir('/scratch/user/uqlbukat/NRAtlas_analysis_2026/NR_code/Pseudobulk')
os.getcwd()

'/scratch/user/uqlbukat/NRAtlas_analysis_2026/NR_code/Pseudobulk'

In [4]:
# Read in the data 
#data = sc.read_h5ad('/scratch/user/uqlbukat/NRAtlas_analysis_2026/20250617_AllDC_bbknn.h5ad')

## Use the NR raw data

In [5]:
# Read in the data 
data = sc.read_h5ad('/scratch/user/uqlbukat/NRAtlas_analysis_2026/NRclean_Raw_Data.h5ad')

In [4]:
data

AnnData object with n_obs × n_vars = 498023 × 15074
    obs: 'nCount_RNA', 'nFeature_RNA', 'sample_type', 'cancer_type', 'patient_id', 'sample_id', 'percent.mt', 'site', 'sample_type_major', 'cancer_subtype', 'integration_id', 'ident', 'dataset_id', 'Technology', 'NR_annotations_simple', 'sample_type_major2', 'cancer_with_H', 'cancer_broadest', 'patient_treatment'
    var: 'name'

In [5]:
# filter to just ovarian cancer
#data = data[data.obs["cancer_broadest"].isin(["ovarian cancer"])]
data_oc = data[data.obs["cancer_with_H"] == "HGSOC"].copy()

In [6]:
data_oc

AnnData object with n_obs × n_vars = 213319 × 15074
    obs: 'nCount_RNA', 'nFeature_RNA', 'sample_type', 'cancer_type', 'patient_id', 'sample_id', 'percent.mt', 'site', 'sample_type_major', 'cancer_subtype', 'integration_id', 'ident', 'dataset_id', 'Technology', 'NR_annotations_simple', 'sample_type_major2', 'cancer_with_H', 'cancer_broadest', 'patient_treatment'
    var: 'name'

# Make pseudobulk dataframes: 

In [7]:
#subset out each DC
data_DC1 = data_oc[data_oc.obs["NR_annotations_simple"].isin(["cDC1"])]
data_DC2 = data_oc[data_oc.obs["NR_annotations_simple"].isin(["cDC2"])]
data_mregDC = data_oc[data_oc.obs["NR_annotations_simple"].isin(["mregDC"])]

In [8]:
#checking how many samples in each (this will have to match metadata dataframe below)
num_unique_integration_ids = data_mregDC.obs['integration_id'].nunique()
print(f"Number of unique integration_id values: {num_unique_integration_ids}")

Number of unique integration_id values: 167


In [9]:
data_DC1_matrix = data_DC1.X.toarray()
data_DC2_matrix = data_DC2.X.toarray()
data_mregDC_matrix = data_mregDC.X.toarray()

In [10]:
genes_DC1 = data_DC1.var_names
obs_DC1 = data_DC1.obs

genes_DC2 = data_DC2.var_names
obs_DC2 = data_DC2.obs

genes_mregDC = data_mregDC.var_names
obs_mregDC = data_mregDC.obs

In [11]:
# Create a DataFrame with cells as rows and genes as columns
data_DC1_df = pd.DataFrame(data_DC1_matrix, index=obs_DC1.index, columns=genes_DC1)
data_DC2_df = pd.DataFrame(data_DC2_matrix, index=obs_DC2.index, columns=genes_DC2)
data_mregDC_df = pd.DataFrame(data_mregDC_matrix, index=obs_mregDC.index, columns=genes_mregDC)

In [12]:
# Add integration_id
data_DC1_df['integration_id'] = obs_DC1['integration_id']
data_DC2_df['integration_id'] = obs_DC2['integration_id']
data_mregDC_df['integration_id'] = obs_mregDC['integration_id']

In [13]:
data_DC1_df.head()

,FAM87B,LINC00115,FAM41C,SAMD11,NOC2L,KLHL17,PLEKHN1,HES4,ISG15,AGRN,...,MT-ATP8,MT-ATP6,MT-CO3,MT-ND3,MT-ND4L,MT-ND4,MT-ND5,MT-ND6,MT-CYB,integration_id
PRJCA005422_EOC1_FS_cell_AAACCTGGTCGGCTCA,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,...,7.0,11.0,29.0,15.0,23.0,5.0,3.0,0.0,24.0,PRJCA005422_HGSOC1_AS
PRJCA005422_EOC1_FS_cell_AAACCTGTCATCATTC,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,15.0,26.0,44.0,19.0,20.0,7.0,12.0,4.0,32.0,PRJCA005422_HGSOC1_AS
PRJCA005422_EOC1_FS_cell_AAAGTAGCATCCGCGA,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,3.0,0.0,...,13.0,20.0,43.0,17.0,36.0,5.0,4.0,0.0,33.0,PRJCA005422_HGSOC1_AS
PRJCA005422_EOC1_FS_cell_AAATGCCAGGCAGGTT,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,18.0,33.0,48.0,39.0,19.0,16.0,11.0,1.0,19.0,PRJCA005422_HGSOC1_AS
PRJCA005422_EOC1_FS_cell_AACCATGCAATGGATA,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,25.0,71.0,83.0,61.0,60.0,26.0,27.0,7.0,98.0,PRJCA005422_HGSOC1_AS


In [14]:
# Group by integration_id and sum to pseudobulk
pseudobulk_DC1_df = data_DC1_df.groupby('integration_id').sum().transpose()
pseudobulk_DC2_df = data_DC2_df.groupby('integration_id').sum().transpose()
pseudobulk_mregDC_df = data_mregDC_df.groupby('integration_id').sum().transpose()

/scratch/temp/24618051/ipykernel_1090934/1587600822.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  pseudobulk_DC1_df = data_DC1_df.groupby('integration_id').sum().transpose()
/scratch/temp/24618051/ipykernel_1090934/1587600822.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  pseudobulk_DC2_df = data_DC2_df.groupby('integration_id').sum().transpose()
/scratch/temp/24618051/ipykernel_1090934/1587600822.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the fut

In [15]:
pseudobulk_DC1_df.head()

integration_id,GSE180661_SPECTRUM-OV-002_S1_CD45P_RIGHT_OVARY,GSE180661_SPECTRUM-OV-003_S1_UNSORTED_INFRACOLIC_OMENTUM,GSE180661_SPECTRUM-OV-003_S1_UNSORTED_LEFT_ADNEXA,GSE180661_SPECTRUM-OV-003_S1_UNSORTED_LEFT_UPPER_QUADRANT,GSE180661_SPECTRUM-OV-003_S1_UNSORTED_PELVIC_PERITONEUM,GSE180661_SPECTRUM-OV-003_S1_UNSORTED_RIGHT_ADNEXA,GSE180661_SPECTRUM-OV-003_S1_UNSORTED_RIGHT_UPPER_QUADRANT,GSE180661_SPECTRUM-OV-008_S1_CD45P_BOWEL,GSE180661_SPECTRUM-OV-008_S1_CD45P_INFRACOLIC_OMENTUM,GSE180661_SPECTRUM-OV-009_S1_CD45P_BOWEL,...,PRJCA005422_HGSOC4_PT,PRJCA005422_HGSOC5_AS,PRJCA005422_HGSOC6_AS,PRJCA005422_HGSOC6_MT,PRJCA005422_HGSOC7_PT,PRJCA005422_HGSOC8_AS,PRJCA005422_HGSOC8_PT,PRJCA005422_HGSOC9_AS,PRJCA005422_HGSOC9_PT,PRJCA005422_HGSOC10_AS
FAM87B,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
LINC00115,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,...,1.0,0.0,0.0,0.0,0.0,2.0,0.0,5.0,0.0,1.0
FAM41C,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
SAMD11,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
NOC2L,5.0,4.0,1.0,1.0,3.0,0.0,2.0,3.0,0.0,7.0,...,3.0,21.0,2.0,1.0,2.0,9.0,4.0,28.0,1.0,16.0


In [16]:
pseudobulk_DC2_df.head()

integration_id,GSE180661_SPECTRUM-OV-002_S1_CD45P_INFRACOLIC_OMENTUM,GSE180661_SPECTRUM-OV-002_S1_CD45P_RIGHT_OVARY,GSE180661_SPECTRUM-OV-003_S1_UNSORTED_INFRACOLIC_OMENTUM,GSE180661_SPECTRUM-OV-003_S1_UNSORTED_LEFT_ADNEXA,GSE180661_SPECTRUM-OV-003_S1_UNSORTED_LEFT_UPPER_QUADRANT,GSE180661_SPECTRUM-OV-003_S1_UNSORTED_PELVIC_PERITONEUM,GSE180661_SPECTRUM-OV-003_S1_UNSORTED_RIGHT_ADNEXA,GSE180661_SPECTRUM-OV-003_S1_UNSORTED_RIGHT_UPPER_QUADRANT,GSE180661_SPECTRUM-OV-008_S1_CD45P_BOWEL,GSE180661_SPECTRUM-OV-008_S1_CD45P_INFRACOLIC_OMENTUM,...,PRJCA005422_HGSOC6_AS,PRJCA005422_HGSOC6_MT,PRJCA005422_HGSOC6_PT,PRJCA005422_HGSOC7_PT,PRJCA005422_HGSOC8_AS,PRJCA005422_HGSOC8_PT,PRJCA005422_HGSOC9_AS,PRJCA005422_HGSOC9_PT,PRJCA005422_HGSOC10_AS,PRJCA005422_HGSOC10_PT
FAM87B,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
LINC00115,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,2.0,...,6.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
FAM41C,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
SAMD11,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
NOC2L,2.0,7.0,3.0,0.0,3.0,0.0,1.0,1.0,0.0,13.0,...,20.0,3.0,3.0,7.0,1.0,0.0,3.0,2.0,8.0,3.0


In [ ]:
pseudobulk_mregDC_df.head()

NameError: name 'pseudobulk_mregDC_df' is not defined

In [21]:
os.getcwd()

'/scratch/user/uqlbukat/NRAtlas_analysis_2026/NR_code/Pseudobulk'

In [22]:
# save as csv
pseudobulk_DC1_df.to_csv("oc_pseudobulk_DC1_data.csv", index=True)
pseudobulk_DC2_df.to_csv("oc_pseudobulk_DC2_data.csv", index=True)
pseudobulk_mregDC_df.to_csv("oc_pseudobulk_mregDC_data.csv", index=True)

# Make metadata dataframes

In [18]:
data_DC1

View of AnnData object with n_obs × n_vars = 2371 × 15074
    obs: 'nCount_RNA', 'nFeature_RNA', 'sample_type', 'cancer_type', 'patient_id', 'sample_id', 'percent.mt', 'site', 'sample_type_major', 'cancer_subtype', 'integration_id', 'ident', 'dataset_id', 'Technology', 'NR_annotations_simple', 'sample_type_major2', 'cancer_with_H', 'cancer_broadest', 'patient_treatment'
    var: 'name'

In [19]:
metadata_DC1 = data_DC1.obs[['integration_id', 'sample_type_major2']].groupby('integration_id').first()
metadata_DC2 = data_DC2.obs[['integration_id', 'sample_type_major2']].groupby('integration_id').first()
metadata_mregDC = data_mregDC.obs[['integration_id', 'sample_type_major2']].groupby('integration_id').first()

/scratch/temp/24618051/ipykernel_1090934/4277918564.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  metadata_DC1 = data_DC1.obs[['integration_id', 'sample_type_major2']].groupby('integration_id').first()
/scratch/temp/24618051/ipykernel_1090934/4277918564.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  metadata_DC2 = data_DC2.obs[['integration_id', 'sample_type_major2']].groupby('integration_id').first()
/scratch/temp/24618051/ipykernel_1090934/4277918564.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to r

In [20]:
metadata_mregDC.head()

,sample_type_major2
integration_id,
GSE180661_SPECTRUM-OV-002_S1_CD45P_INFRACOLIC_OMENTUM,metastatic tumour
GSE180661_SPECTRUM-OV-002_S1_CD45P_RIGHT_OVARY,primary tumour
GSE180661_SPECTRUM-OV-003_S1_UNSORTED_INFRACOLIC_OMENTUM,metastatic tumour
GSE180661_SPECTRUM-OV-003_S1_UNSORTED_LEFT_ADNEXA,primary tumour
GSE180661_SPECTRUM-OV-003_S1_UNSORTED_LEFT_UPPER_QUADRANT,metastatic tumour


In [21]:
metadata_mregDC.shape

(167, 1)

In [22]:
# set the index (integration_id) to be a column
metadata_DC1 = metadata_DC1.reset_index()
metadata_DC2 = metadata_DC2.reset_index()
metadata_mregDC = metadata_mregDC.reset_index()

In [23]:
metadata_mregDC.head()

,integration_id,sample_type_major2
0,GSE180661_SPECTRUM-OV-002_S1_CD45P_INFRACOLIC_...,metastatic tumour
1,GSE180661_SPECTRUM-OV-002_S1_CD45P_RIGHT_OVARY,primary tumour
2,GSE180661_SPECTRUM-OV-003_S1_UNSORTED_INFRACOL...,metastatic tumour
3,GSE180661_SPECTRUM-OV-003_S1_UNSORTED_LEFT_ADNEXA,primary tumour
4,GSE180661_SPECTRUM-OV-003_S1_UNSORTED_LEFT_UPP...,metastatic tumour


In [29]:
# save as csv
metadata_DC1.to_csv("oc_pseudobulk_sampletype_metadata_DC1.csv", index=False)
metadata_DC2.to_csv("oc_pseudobulk_sampletype_metadata_DC2.csv", index=False)
metadata_mregDC.to_csv("oc_pseudobulk_sampletype_metadata_mregDC.csv", index=False)